# Singlish Whisper LoRA Development Sandbox

A parameter exploration notebook for rapid iteration on `whisper-large-v3-turbo` with LoRA.

Download this notebook, tweak hyperparameters in the configuration cell below, run the training and evaluation, and observe test WER, CER, and term recall before committing changes back to `src/` or `scripts/`.

### Prerequisites
1. Clone this repository locally.
2. Run `.\scripts\setup.ps1` in PowerShell to create `.venv` with the CUDA 12.8 PyTorch wheel (for RTX 50-series / Blackwell and earlier NVIDIA GPUs).
3. Select `.venv` as your Jupyter Notebook kernel.

In [ ]:
import os
import sys
from pathlib import Path
import torch

os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")

# Ensure working directory is the repo root
ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
    if str(ROOT) not in sys.path:
        sys.path.insert(0, str(ROOT))
    os.chdir(ROOT)

print("Python:", sys.executable)
print("Workspace root:", ROOT)
print("PyTorch:", torch.__version__, "| CUDA:", torch.version.cuda)
if torch.cuda.is_available():
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print("GPU:", torch.cuda.get_device_name(0), f"({vram_gb:.1f} GB)")
else:
    print("Warning: No CUDA GPU detected. Training will run on CPU and take considerable time.")

from src.config import settings
print("HF token configured:", bool(settings.hf_token))
print("OpenAI API key configured:", bool(settings.openai_api_key))

In [ ]:
# ==============================================================================
# EXPERIMENT HYPERPARAMETERS (Tweak these to test different training configs)
# ==============================================================================

# Data Preparation
REBUILD_DATA = False     # Set to True to re-download SASRBench-v1 and run VAD preprocessing
DATA_LIMIT = None        # Cap downloaded/preprocessed clips (e.g. 32 for quick dry run, None for full 3,747)

# Training Hyperparameters
OUTPUT_DIR = "checkpoints/dev-lora"  # Dev path to avoid overwriting production checkpoints
EPOCHS = 3.0                         # Number of training epochs (e.g., 1.0, 3.0, 5.0)
MAX_STEPS = 0                        # Overrides epochs if > 0 (e.g. 50 for a fast parameter smoke test)
BATCH_SIZE = 8                       # Per-device batch size (reduce to 4 or 2 if VRAM is constrained)
GRAD_ACCUM = 2                       # Gradient accumulation steps (effective batch size = BATCH_SIZE * GRAD_ACCUM)
LEARNING_RATE = 1e-3                 # LoRA learning rate (typical sweep: 5e-4, 1e-3, 2e-3)
WARMUP_STEPS = 50                    # Learning rate linear warmup steps
EVAL_STEPS = 100                     # Checkpoint evaluation frequency in steps

# LoRA Architecture
LORA_R = 32                          # LoRA rank dimension (typical sweep: 16, 32, 64)
LORA_ALPHA = 64                      # LoRA scaling factor (conventionally 2 * LORA_R)

# Dataset Slicing
TRAIN_LIMIT = None                   # Cap training clips (e.g. 32 for smoke testing, None for full 2,946 clips)
EVAL_LIMIT = None                    # Cap evaluation clips (e.g. 16 for smoke testing, None for full 390 test clips)

print(f"Configured run -> Output: {OUTPUT_DIR} | Epochs: {EPOCHS} | Batch: {BATCH_SIZE}x{GRAD_ACCUM} | LR: {LEARNING_RATE} | Rank: {LORA_R}")

In [ ]:
import subprocess
import sys
from pathlib import Path

train_manifest = Path("data/manifests/train.jsonl")

if REBUILD_DATA or not train_manifest.is_file():
    print("Running data download, VAD preprocessing, and manifest creation...")
    slice_args = ["--limit", str(DATA_LIMIT)] if DATA_LIMIT else []
    subprocess.run([sys.executable, "-m", "src.data_pipeline.download", *slice_args], check=True)
    subprocess.run([sys.executable, "-m", "src.data_pipeline.ray_preprocess", *slice_args], check=True)
    subprocess.run([sys.executable, "-m", "src.data_pipeline.manifest"], check=True)
else:
    print(f"Existing manifests detected in data/manifests/ (REBUILD_DATA={REBUILD_DATA}). Skipping data rebuild.")

In [ ]:
import subprocess
import sys

train_cmd = [
    sys.executable, "-m", "src.training.train",
    "--output-dir", OUTPUT_DIR,
    "--epochs", str(EPOCHS),
    "--max-steps", str(MAX_STEPS),
    "--batch-size", str(BATCH_SIZE),
    "--grad-accum", str(GRAD_ACCUM),
    "--lr", str(LEARNING_RATE),
    "--warmup-steps", str(WARMUP_STEPS),
    "--eval-steps", str(EVAL_STEPS),
    "--lora-r", str(LORA_R),
    "--lora-alpha", str(LORA_ALPHA),
]

if TRAIN_LIMIT is not None:
    train_cmd += ["--limit", str(TRAIN_LIMIT)]

print("Launching training with command:")
print(" ".join(train_cmd))
subprocess.run(train_cmd, check=True)

In [ ]:
import subprocess
import sys
from pathlib import Path

eval_limit_args = ["--limit", str(EVAL_LIMIT)] if EVAL_LIMIT else []

print("1. Evaluating base Whisper model (zero-shot baseline)...")
subprocess.run([sys.executable, "-m", "src.eval.run_eval", "--variant", "base", *eval_limit_args], check=True)

adapter_cfg = Path(OUTPUT_DIR) / "adapter_config.json"
if adapter_cfg.is_file():
    print(f"2. Evaluating fine-tuned dev adapter from {OUTPUT_DIR}...")
    subprocess.run(
        [sys.executable, "-m", "src.eval.run_eval", "--variant", "finetuned", "--adapter-path", OUTPUT_DIR, *eval_limit_args],
        check=True,
    )
else:
    print(f"No adapter found at {OUTPUT_DIR}. Training did not save an adapter.")

print("3. Generating comparison table...")
subprocess.run([sys.executable, "-m", "src.eval.run_eval", "--compare"], check=True)

comparison_file = Path("data/eval/comparison.md")
if comparison_file.is_file():
    print("\n" + comparison_file.read_text(encoding="utf-8"))

## Promoting Tweaked Parameters to the Main Codebase

If your experimental parameters produce superior WER, CER, or Singlish term recall:

1. **Update default CLI arguments in `src/training/train.py`:**
   - Update `--epochs`, `--batch-size`, `--grad-accum`, `--lr`, `--lora-r`, `--lora-alpha`.
2. **Update pipeline scripts:**
   - Modify `scripts/run_full_pipeline.ps1` with the tested values.
3. **Train the production checkpoint:**
   ```powershell
   python -m src.training.train --output-dir checkpoints/whisper-turbo-singlish-lora
   python -m src.eval.run_eval --variant base
   python -m src.eval.run_eval --variant finetuned --adapter-path checkpoints/whisper-turbo-singlish-lora
   python -m src.eval.run_eval --compare
   ```
4. **Optional - Run hotword mining & biasing:**
   ```powershell
   python -m src.eval.decode --manifests data/manifests/train.jsonl data/manifests/val.jsonl --variant base --out data/hyps/base_trainval.jsonl
   python -m src.lexicon.mine_hotwords --hyps data/hyps/base_trainval.jsonl
   python -m src.lexicon.llm_curate
   python -m src.eval.run_eval --variant finetuned_biased --adapter-path checkpoints/whisper-turbo-singlish-lora
   ```